Universidade do Vale do Itajaí<br>
PPGCA - Programa de Pós-Graduação em Computação Aplicada<br>
Aprendizado Profundo<br>
Prof. Felipe Viel<br>
Conteúdo: Redes Neurais Transformer<br>

Ao final do exercício, você deverá ser capaz de:

1. Explicar a ideia de self-attention e o papel de Query, Key e Value.
2. Implementar um Transformer Encoder pequeno usando TensorFlow/Keras.
3. Comparar uma rede LSTM com um Transformer em uma série temporal.
4. Entender como uma imagem pode ser transformada em uma sequência de *patches*.
5. Implementar um Vision Transformer (ViT) simplificado.
6. Avaliar os modelos por métricas, curvas de treinamento e exemplos de predição.
7. Identificar situações em que Transformer, LSTM ou CNN podem ser mais adequados.

> **Referências didáticas:** o primeiro bloco é inspirado na explicação visual de Jay Alammar sobre Transformer, especialmente a decomposição em embeddings, positional encoding, self-attention, multi-head attention e feed-forward. O exercício de séries temporais segue a lógica de comparação com redes recorrentes presente no notebook de Redes Recorrentes fornecido como exercício de RNN. O bloco de visão utiliza o CIFAR-10 como ponto de partida, seguindo o notebook de Redes Convolucionais fornecido como exercício de CNN.

Preparação do ambiente

### TensorFlow ou PyTorch?

Neste notebook, a implementação principal será feita em **TensorFlow/Keras**, pois permite construir o modelo de maneira relativamente compacta.

Em PyTorch, os conceitos são equivalentes. As principais correspondências são:

| TensorFlow/Keras | PyTorch |
|---|---|
| `tf.keras.layers.MultiHeadAttention` | `torch.nn.MultiheadAttention` |
| `tf.keras.layers.Embedding` | `torch.nn.Embedding` |
| `tf.keras.layers.LayerNormalization` | `torch.nn.LayerNorm` |
| `model.fit()` | loop de treinamento / Lightning |
| `tf.data.Dataset` | `torch.utils.data.DataLoader` |

Dica: não tente decorar a API. O mais importante é compreender o fluxo dos tensores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# PARTE I — Caso-base: entendendo o Transformer

## 1. Do texto ao Transformer

No artigo *The Illustrated Transformer*, a arquitetura original é apresentada como uma combinação de:

- embeddings;
- informação de posição;
- self-attention;
- multi-head attention;
- redes feed-forward;
- conexões residuais;
- normalização;
- encoder e decoder.

A ideia central do **self-attention** pode ser resumida por:

$$
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Onde:

- Q (Query): o que uma posição está procurando;
- K (Key): como cada posição pode ser identificada para comparação;
- V (Value): informação que será agregada;
- $d_k$: dimensão das chaves.

No exemplo clássico do artigo, o Transformer é apresentado no contexto de tradução. Para tornar o exercício executável em uma disciplina, vamos usar a mesma ideia estrutural em um problema mais simples: classificação de textos.

Fonte conceitual: [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/).

## 2. Primeiro experimento: visualizar a atenção manualmente

Antes de usar uma camada pronta do Keras, vamos construir uma versão mínima do cálculo de atenção.

Exercício guiado: observe as dimensões de `Q`, `K`, `V`, dos scores e da matriz final de atenção.

In [ ]:
# Exemplo pequeno: 4 tokens, representação com 8 dimensões
X = tf.random.normal((1, 4, 8))

Wq = tf.random.normal((8, 4))
Wk = tf.random.normal((8, 4))
Wv = tf.random.normal((8, 4))

Q = X @ Wq
K = X @ Wk
V = X @ Wv

scores = tf.matmul(Q, K, transpose_b=True)
scores = scores / tf.sqrt(tf.cast(tf.shape(K)[-1], tf.float32))

weights = tf.nn.softmax(scores, axis=-1)
Z = tf.matmul(weights, V)

print("X:", X.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)
print("Attention weights:", weights.shape)
print("Saída Z:", Z.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(weights[0].numpy(), cmap="viridis")
plt.colorbar()
plt.xlabel("Key")
plt.ylabel("Query")
plt.title("Matriz de pesos de self-attention")
plt.show()

### Exercício 1 — Investigação

Modifique o código anterior e responda:

1. O que acontece com `weights` quando os valores de `scores` ficam muito grandes?
2. Por que utilizamos a divisão por $\sqrt{d_k}$?
3. Qual dimensão representa a quantidade de tokens?
4. O que significa uma linha da matriz de atenção?
5. Faça uma alteração para aumentar o número de tokens de 4 para 8.

**Entrega:** registre suas respostas em uma célula Markdown.

## 3. Positional Encoding

Self-attention, isoladamente, não possui uma noção natural de ordem. Para uma sequência, precisamos fornecer ao modelo alguma informação sobre a posição dos elementos.

Uma possibilidade é utilizar codificação posicional senoidal:

$$
PE(pos,2i)=sin\left(pos/10000^{2i/d}\right)
$$

$$
PE(pos,2i+1)=cos\left(pos/10000^{2i/d}\right)
$$

Não é necessário implementar a versão completa para o primeiro experimento. Vamos criar uma camada simples de posição aprendida.

In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab_size, max_length, d_model):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(max_length, d_model)

    def call(self, x):
        length = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

## 4. Transformer Encoder mínimo

A unidade fundamental utilizada neste exercício será:

**entrada → multi-head self-attention → residual + normalização → feed-forward → residual + normalização**

Isso corresponde à ideia central do encoder apresentado no *Illustrated Transformer*. 

O objetivo aqui não é reproduzir literalmente o Transformer original de tradução, mas construir uma versão pequena o suficiente para que o estudante consiga acompanhar as dimensões e o fluxo dos dados.

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model)
        ])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn = self.att(x, x)
        x = self.norm1(x + self.drop1(attn, training=training))
        ffn = self.ffn(x)
        return self.norm2(x + self.drop2(ffn, training=training))

# PARTE II — Aplicação em texto

Vamos utilizar o dataset **IMDB**, contendo avaliações de filmes classificadas em duas classes. O objetivo é prever se uma avaliação é positiva ou negativa.

O estudante deve observar que o problema é diferente do exemplo de tradução do artigo, mas os componentes fundamentais do encoder continuam presentes.

### Exercício 2 — Construção do classificador

Complete e execute o modelo abaixo.

**Tarefa:** alterar pelo menos dois hiperparâmetros e comparar o resultado:

- `d_model`;
- número de cabeças;
- dimensão do feed-forward;
- número de blocos Transformer;
- comprimento máximo da sequência.

In [ ]:
vocab_size = 10000
max_length = 200
d_model = 64
num_heads = 4
ff_dim = 128

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=vocab_size
)

x_train = keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding="post", truncating="post"
)
x_test = keras.preprocessing.sequence.pad_sequences(
    x_test, maxlen=max_length, padding="post", truncating="post"
)

print(x_train.shape, x_test.shape)

In [ ]:
inputs = keras.Input(shape=(max_length,), dtype="int32")

x = PositionalEmbedding(vocab_size, max_length, d_model)(inputs)
x = TransformerEncoder(d_model, num_heads, ff_dim)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

text_transformer = keras.Model(inputs, outputs)
text_transformer.summary()

text_transformer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_text = text_transformer.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128
)

In [ ]:
test_loss, test_acc = text_transformer.evaluate(x_test, y_test, verbose=0)
print(f"Acurácia no teste: {test_acc:.4f}")

plt.figure(figsize=(8,4))
plt.plot(history_text.history["accuracy"], label="treino")
plt.plot(history_text.history["val_accuracy"], label="validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Transformer — classificação de texto")
plt.legend()
plt.show()

### Perguntas para discussão

1. O que muda quando aumentamos `num_heads`?
2. O que acontece se retirarmos a informação posicional?
3. O que representa o vetor produzido para cada token após a self-attention?
4. Por que usamos `GlobalAveragePooling1D` neste classificador?
5. O Transformer utilizado aqui é encoder-only, decoder-only ou encoder-decoder? Justifique.

# PARTE III — Séries temporais: Transformer × LSTM

Agora vamos transportar a ideia de atenção para outro domínio.

No notebook de Redes Recorrentes utilizado como referência, a abordagem consiste em organizar os dados temporais em janelas e utilizar uma rede recorrente para aprender a relação entre passado e futuro.

Aqui faremos a mesma ideia geral, mas substituindo a LSTM por um Transformer Encoder.

### Problema

Será criada uma série temporal sintética contendo tendência, sazonalidade e ruído:

$$
y_t = 0.02t + sin(t/5) + 0.5sin(t/17) + \epsilon
$$

O modelo receberá uma janela de valores anteriores e deverá prever o próximo valor.

Fonte de referência da estrutura de exercícios: notebook de Redes Recorrentes do repositório fornecido pelo professor. 

In [ ]:
t = np.arange(0, 2500)
rng = np.random.default_rng(SEED)

series = (
    0.02 * t
    + np.sin(t / 5)
    + 0.5 * np.sin(t / 17)
    + 0.15 * rng.normal(size=len(t))
)

plt.figure(figsize=(12,4))
plt.plot(t[:500], series[:500])
plt.title("Exemplo de série temporal")
plt.xlabel("Tempo")
plt.ylabel("Valor")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

split = int(len(series) * 0.8)

scaler = StandardScaler()
train_series = scaler.fit_transform(series[:split].reshape(-1,1)).flatten()
test_series = scaler.transform(series[split:].reshape(-1,1)).flatten()

WINDOW = 40

def make_windows(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_series, WINDOW)
X_test, y_test = make_windows(test_series, WINDOW)

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

## 5. Modelo LSTM — linha de base

Primeiro, construa uma linha de base com LSTM. Isso é importante porque não devemos avaliar um Transformer isoladamente: precisamos de uma referência.

A LSTM possui uma dinâmica recorrente explícita. O Transformer, por outro lado, pode relacionar posições da janela por meio da atenção.

In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(WINDOW, 1)),
    layers.LSTM(64),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

In [ ]:
hist_lstm = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)

## 6. Transformer para série temporal

Agora construiremos um Transformer Encoder para a mesma tarefa.

A entrada possui a forma:

`(batch, janela, características)`

No nosso caso:

`(batch, 40, 1)`

A projeção inicial transforma cada valor da série em um vetor de dimensão `d_model`.

In [ ]:
class TimeSeriesTransformer(keras.Model):
    def __init__(self, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
        super().__init__()
        self.proj = layers.Dense(d_model)
        self.encoder = TransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )
        self.pool = layers.GlobalAveragePooling1D()
        self.drop = layers.Dropout(dropout)
        self.out = layers.Dense(1)

    def call(self, x, training=False):
        x = self.proj(x)
        x = self.encoder(x, training=training)
        x = self.pool(x)
        x = self.drop(x, training=training)
        return self.out(x)

ts_transformer = TimeSeriesTransformer()
ts_transformer.compile(optimizer="adam", loss="mse", metrics=["mae"])

# build
_ = ts_transformer(tf.zeros((1, WINDOW, 1)))
ts_transformer.summary()

In [ ]:
hist_trans = ts_transformer.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)

In [ ]:
lstm_pred = lstm_model.predict(X_test, verbose=0).flatten()
trans_pred = ts_transformer.predict(X_test, verbose=0).flatten()

lstm_mse = np.mean((y_test - lstm_pred)**2)
trans_mse = np.mean((y_test - trans_pred)**2)

print("MSE LSTM:", lstm_mse)
print("MSE Transformer:", trans_mse)

In [ ]:
plt.figure(figsize=(12,5))
n = 200
plt.plot(y_test[:n], label="real")
plt.plot(lstm_pred[:n], label="LSTM")
plt.plot(trans_pred[:n], label="Transformer")
plt.title("Previsão da série temporal")
plt.legend()
plt.show()

### Exercício 3 — Investigação experimental

Faça pelo menos **três experimentos** alterando:

- tamanho da janela (`WINDOW`);
- número de cabeças;
- dimensão do embedding;
- número de camadas Transformer;
- número de unidades da LSTM.

Monte uma tabela contendo:

| Modelo | Janela | Parâmetros | MAE | MSE |
|---|---:|---:|---:|---:|
| LSTM | 40 | ... | ... | ... |
| Transformer | 40 | ... | ... | ... |
| LSTM | 80 | ... | ... | ... |
| Transformer | 80 | ... | ... | ... |

Questão: o Transformer sempre apresenta desempenho superior à LSTM? Explique.

# PARTE IV — Vision Transformer

## 7. De imagem para sequência

No notebook de Redes Convolucionais utilizado como referência, o CIFAR-10 é tratado como um problema de classificação de imagens RGB de tamanho $32\times32\times3$. O notebook utiliza convoluções e pooling para extrair características espaciais. 

O Vision Transformer muda a representação:

**Imagem → patches → vetores → embeddings → positional embedding → Transformer Encoder → classificação**

Por exemplo, para uma imagem $32\times32$ usando patches $4\times4$:

$$
N = \frac{32}{4}\times\frac{32}{4}=64
$$

Assim, uma imagem passa a ser representada como uma sequência de **64 tokens visuais**.

Cada patch contém:

$$
4\times4\times3=48
$$

valores de pixel, que serão projetados para uma dimensão `d_model`.

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()

train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

train_labels = train_labels.squeeze()
test_labels = test_labels.squeeze()

class_names = [
    "avião", "automóvel", "pássaro", "gato", "veado",
    "cachorro", "sapo", "cavalo", "navio", "caminhão"
]

print(train_images.shape, test_images.shape)

In [ ]:
plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Implementando a divisão em patches

A operação abaixo transforma:

`(batch, 32, 32, 3)`

em:

`(batch, 64, 48)`

Observe que não estamos usando convolução para gerar os tokens. Estamos simplesmente reorganizando a imagem em pequenos blocos.

In [ ]:
def extract_patches(images, patch_size=4):
    batch = tf.shape(images)[0]
    patches = tf.image.extract_patches(
        images=images,
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, patch_size, patch_size, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(patches, [batch, -1, patch_size * patch_size * 3])

patches = extract_patches(train_images[:8], patch_size=4)
print("Patches:", patches.shape)

In [ ]:
plt.figure(figsize=(10,3))
for i in range(8):
    patch = patches[0, i].numpy().reshape(4,4,3)
    plt.subplot(2,4,i+1)
    plt.imshow(patch)
    plt.title(f"Patch {i}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Patch embedding + Transformer

Agora cada patch será transformado em um vetor de dimensão `d_model`.

Também adicionaremos um token `[CLS]`, cuja representação final será utilizada para classificação.

In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, d_model):
        super().__init__()
        self.projection = layers.Dense(d_model)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1,
            output_dim=d_model
        )
        self.cls_token = self.add_weight(
            shape=(1, 1, d_model),
            initializer="zeros",
            trainable=True,
            name="cls_token"
        )

    def call(self, patches):
        batch_size = tf.shape(patches)[0]
        x = self.projection(patches)

        cls = tf.broadcast_to(
            self.cls_token,
            [batch_size, 1, tf.shape(x)[-1]]
        )

        x = tf.concat([cls, x], axis=1)

        positions = tf.range(start=0, limit=tf.shape(x)[1])
        return x + self.position_embedding(positions)

In [ ]:
def build_vit(
    image_size=32,
    patch_size=4,
    d_model=64,
    num_heads=4,
    ff_dim=128,
    num_layers=2,
    num_classes=10
):
    num_patches = (image_size // patch_size) ** 2

    inputs = keras.Input(shape=(image_size, image_size, 3))

    x = layers.Lambda(
        lambda z: extract_patches(z, patch_size)
    )(inputs)

    x = PatchEncoder(num_patches, d_model)(x)

    for _ in range(num_layers):
        x = TransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=0.1
        )(x)

    # Primeiro token = CLS
    x = x[:, 0]
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="TinyViT")

vit = build_vit()
vit.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

vit.summary()

### Exercício 4 — Treinar o Vision Transformer

Para tornar o exercício viável em uma aula, o modelo abaixo é deliberadamente pequeno.

Treine por algumas épocas e registre:

- acurácia de treinamento;
- acurácia de validação;
- tempo de treinamento;
- quantidade de parâmetros.

In [ ]:
hist_vit = vit.fit(
    train_images, train_labels,
    validation_split=0.1,
    epochs=5,
    batch_size=128
)

In [ ]:
vit_loss, vit_acc = vit.evaluate(test_images, test_labels, verbose=0)
print(f"ViT — acurácia no teste: {vit_acc:.4f}")

plt.figure(figsize=(8,4))
plt.plot(hist_vit.history["accuracy"], label="treino")
plt.plot(hist_vit.history["val_accuracy"], label="validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Vision Transformer — CIFAR-10")
plt.legend()
plt.show()

## 10. Comparação conceitual: CNN × ViT

O objetivo não é concluir que uma arquitetura é sempre melhor.

No exercício original com CIFAR-10, a CNN explora diretamente a estrutura local da imagem por meio de filtros convolucionais e pooling. 

No ViT, a imagem é convertida em uma sequência de patches e o Transformer aprende relações entre esses tokens.

### Responda

1. Quantos tokens são gerados para patch $4\times4$?
2. Quantos tokens seriam gerados com patch $8\times8$?
3. O que acontece com o custo computacional quando o número de patches aumenta?
4. Qual informação é perdida quando transformamos uma imagem em patches?
5. Por que precisamos de positional embeddings?
6. O token `[CLS]` representa o quê?
7. Por que um ViT pequeno pode apresentar desempenho inferior a uma CNN em um dataset pequeno?

# PARTE V — Experimento final integrador

Agora você deverá escolher uma das três aplicações e modificar a arquitetura.

## Opção A — Texto

Experimente:

- 2 ou 4 blocos Transformer;
- 2, 4 ou 8 cabeças;
- diferentes tamanhos de embedding.

Compare acurácia e número de parâmetros.

## Opção B — Série temporal

Experimente:

- janelas de 20, 40 e 80 pontos;
- LSTM versus Transformer;
- uma camada versus duas camadas Transformer.

Compare MAE e MSE.

## Opção C — Visão

Experimente:

- patch $4\times4$ versus $8\times8$;
- 2 versus 4 cabeças;
- 1 versus 2 blocos Transformer.

Compare acurácia, parâmetros e tempo de treinamento.

# PARTE VI — Desafio opcional: enxergar a atenção

Uma característica interessante dos Transformers é a possibilidade de analisar os pesos de atenção.

Desafio: modifique `TransformerEncoder` para retornar também `attention_scores=True` na camada `MultiHeadAttention`.

Depois:

1. extraia os pesos de atenção;
2. visualize-os como uma matriz;
3. discuta quais tokens parecem receber maior atenção.

> Atenção: uma matriz de atenção não deve ser interpretada automaticamente como uma explicação causal do modelo. Ela é uma ferramenta de inspeção do comportamento interno.

# PARTE VII — Implementação equivalente em PyTorch

Se você preferir PyTorch, a estrutura conceitual pode ser traduzida aproximadamente assim:

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, ff_dim):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, weights = self.attn(x, x, x)
        x = self.norm1(x + attn_out)

        x = self.norm2(x + self.ffn(x))

        return x, weights
```

A lógica permanece a mesma:

**tokens → embeddings → posição → atenção → feed-forward → residual → normalização → classificação**

No PyTorch, lembre-se de conferir sempre a convenção das dimensões dos tensores e o parâmetro `batch_first=True`.

# Produza para conhecimento próprio (ou para usar no Trabalho Final)

O notebook deve ser entregue com:

### 1. Código
- células executadas;
- modelos treinados;
- alterações realizadas nos hiperparâmetros.

### 2. Resultados
Apresente uma tabela:

| Aplicação | Modelo | Parâmetros | Métrica principal | Resultado |
|---|---|---:|---|---:|
| Texto | Transformer | ... | Accuracy | ... |
| Série | LSTM | ... | MAE/MSE | ... |
| Série | Transformer | ... | MAE/MSE | ... |
| Imagem | CNN | ... | Accuracy | ... |
| Imagem | ViT | ... | Accuracy | ... |

### 3. Discussão

Responda, em texto, às seguintes questões:

1. Qual é a principal diferença entre processamento recorrente e self-attention?
2. Por que o Transformer consegue processar diferentes posições de uma sequência de maneira paralela?
3. Por que informação posicional é necessária?
4. O que muda quando aumentamos o número de cabeças?
5. Qual foi o efeito do tamanho da janela na série temporal?
6. Qual foi o efeito do tamanho dos patches no ViT?
7. Em qual dos três problemas o Transformer apresentou maior vantagem?
8. Em qual problema uma arquitetura mais simples pareceu suficiente?
9. Qual arquitetura você escolheria para um novo problema e por quê?

